# Learning Objectives

* Build a per-sample RNA-Seq workflow (QC, alignment, counting, normalization, differential expression) from Python wrappers and an Apptainer container.
* Generate and register FASTQ inputs, references, R support scripts, a container, and a workflow DAG through the Pegasus Python API.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the results: alignments, BigWig tracks, count tables, PCA and DESeq2 volcano plots.

# Bacterial RNA-Seq Workflow

This workflow analyses bacterial RNA-Seq data, converted from the chienlab-rnaseq Nextflow DSL2 pipeline.
Each sample is trimmed with fastp, aligned with BWA-MEM and converted to a BigWig track in parallel; counts are
then gathered with Rsubread featureCounts, TMM-normalised with edgeR, summarised with a PCA, and — when a
contrast table is given — tested for differential expression with DESeq2. See `README.md` in this repository
for the full pipeline description and outputs.

**Workflow Jobs:**
1. `fastp` - QC and adapter trimming (per sample)
2. `bwa_index` - build the BWA index of the reference genome
3. `bwa_align` - align reads with BWA-MEM (per sample)
4. `bam2bigwig` - BigWig coverage tracks (per sample)
5. `count_reads` - gene-level counts with Rsubread featureCounts (fan-in)
6. `tmm_normalise` - TMM normalisation (edgeR)
7. `pca` - principal component analysis of the samples
8. `diffexpr` - differential expression with DESeq2 (only with a contrast table)

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Prepare Sample Input Data

The workflow needs a tab-separated sample sheet and the FASTQ files it names (see "Input Files" in
`README.md`); the reference genome (`references/NC_011916.fasta`, *C. crescentus* NA1000) and its annotation
(`references/ccna.gff`) ship with the repository. Put the sample sheet at `samples.tsv`, the FASTQs in
`fastq/`, and optionally a contrast table at `contrasts.tsv`, then check them:

In [ ]:
!head -5 samples.tsv && ls fastq/ | head && cat contrasts.tsv

## 2. Build the Apptainer Container

The workflow's jobs run inside an Apptainer container (`Apptainer/RNASeq_Container.def`) that bundles fastp,
BWA, samtools, deepTools and R with Rsubread, edgeR and DESeq2. Building it can take a while; skip the rebuild
if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/RNASeq_Container.sif ]; then
    apptainer build Apptainer/RNASeq_Container.sif Apptainer/RNASeq_Container.def
else
    echo "Container already built: Apptainer/RNASeq_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses `workflow_generator.py` — the same generator used by the command-line tool — to
build the Pegasus workflow DAG (site, transformation, and replica catalogs) and write it to `workflow.yml`.
The inputs are parsed by the CLI's own argument parser (`build_parser()`), so nothing about the workflow is
defined here: change the pipeline in `workflow_generator.py`, and both the notebook and the CLI pick it up.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import RnaseqWorkflow, build_parser

EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The same arguments the CLI takes (see README.md)
args = build_parser().parse_args([
    "--sample-file", "samples.tsv", "--data-dir", "fastq",
    "--ref-genome", "references/NC_011916.fasta",
    "--ref-ann", "references/ccna.gff",
    "--contrast-table", "contrasts.tsv",
    "--output", "workflow.yml",
])

workflow = RnaseqWorkflow(args, dagfile=args.output)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(exec_site_name=EXEC_SITE,
                                       container_sif=args.container_sif)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating RNA-Seq workflow DAG...")
workflow.create_workflow()

workflow.write()
print("\nRNA-Seq Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. The graph shows one
fastp → bwa_align → bam2bigwig chain per sample sharing a single `bwa_index`, fanning in to `count_reads`,
then `tmm_normalise`, `pca` and `diffexpr`.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the site chosen above (`EXEC_SITE`).

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and
prints the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/`:

| Directory/File | Description |
|----------------|-------------|
| `fastp_qc/*_fastp.html` | Per-sample fastp QC reports |
| `bwa_aln/*.bam`, `*.bam.bai` | Per-sample sorted alignments |
| `bigwig/*.bw` | Per-sample BigWig files for a genome browser |
| `read_counts/gene_counts.tsv`, `gene_counts_pc.tsv`, `counts_summary.tsv` | Raw gene-level counts and library composition |
| `read_counts/cpm_counts.tsv`, `rpkm_counts.tsv` | TMM-normalised CPM / RPKM values |
| `read_counts/library_composition*.png` | Library composition plots |
| `PCA_samples/pca_grouped.png`, `pca_coords.tsv` | PCA plot and coordinates |
| `diff_expr/DGE_*.tsv`, `volcano_plot_*.png` | Differential expression results and volcano plots per contrast |

In [ ]:
!ls -ltR output/

Let's look at the PCA of the samples, coloured by group:

In [ ]:
Image(filename="output/PCA_samples/pca_grouped.png")